# Kapitel 5

## Frågor

### 1. Vad menas med curse of dimensionality.

As the number of features (dimensions) grows, the data gets spread more thinly over the feature space, and several things make ML harder.

In high dimensions points tend to be far from each other, and the distances between points become more similar, which hurts methods that rely on distance, like KNN. The amount of data needed to cover the space grows very fast with the number of dimensions, so with a fixed dataset the data becomes sparser, and it gets easier for a model to find random patterns and overfit. So more features isn't automatically better, unless there is also enough data.

### 2. Vad är dimensionsreducering och varför görs det?

Dimensionality reduction means transforming data with many features into fewer features, while keeping as much of the useful information as possible.

Why it's done:
- It reduces the curse of dimensionality (question 1): less sparse data and less risk of overfitting.
- Faster training and prediction with fewer features.
- Visualization: data can be reduced to 2-3 dimensions and plotted.
- It can remove noise, and dropping redundant directions sometimes improves performance.
- Less memory and storage for large datasets.

The downside is that some information and interpretability is lost, so it's a tradeoff.

### 3. Förklara översiktligt hur PCA fungerar. Använd figur 5.4 på sidan 224 i din förklaring.

Figure 5.4 shows a 2D dataset (x1, x2) where the points form an elongated cloud along a diagonal. Three candidate lines go through the data: a solid line along the direction where the data is most spread out (marked c1), a dashed line, and a dotted line perpendicular to c1 (marked c2). The three panels on the right show what the data looks like after projecting every point onto each of the lines, so the 2D data becomes 1D (z1).

The projection onto the solid line (top panel) is the most spread out, the dashed line gives less spread, and the dotted line (bottom panel) squeezes the points into a very narrow range.

That's the idea behind PCA: it picks the direction where the projected data has the largest variance. That direction is the first principal component, c1. The second principal component, c2, is perpendicular to c1 and captures the variance that is left. To reduce this dataset from 2D to 1D, we project every point onto c1 and keep only that coordinate, and lose the small amount of spread in the c2 direction.

Worth noting: PCA keeps as much variance in the features as possible, but it doesn't look at the target at all. So the directions it keeps aren't guaranteed to be the most useful ones for predicting something.

### 5. Stina påstår att man i maskininlärning alltid vill ha modeller som genomför så bra prediktioner som möjligt. Kalle påstår att det inte riktigt stämmer eftersom tid också är en viktig aspekt. Både för själva modellträningen och för själva prediktionerna. Vad säger du?

Kalle has a point. Prediction quality isn't the only thing that matters in practice, time and resources are real constraints too.

Training time matters because if tuning a complex model takes days instead of hours, there's less time to experiment, which can end up hurting quality too. Prediction time often matters even more: if a model has to answer in real time, like fraud checks during a card payment or a self-driving car reacting to something on the road, a slightly less accurate but much faster model is often the right choice. A model that is too slow to use in production doesn't help, however accurate it is.

So it's a tradeoff between accuracy and speed or cost (related to the principle of parsimony from chapter 1). The best model is the one that fits the constraints of the use case, not necessarily the most accurate one. Dimensionality reduction is one tool for this tradeoff, faster training and prediction at the cost of some accuracy.

### 6. Efter att vi genomfört en PCA, vad händer med tolkningen av variablerna?

The components are harder to interpret than the original features. Each principal component is a linear combination of all the original features, so "component 1" doesn't correspond to one real-world variable like "square meters" anymore, but to something like 0.3·square_meters + 0.5·rooms − 0.2·age + ...

So it's much harder to explain what drives a prediction after PCA. We can't point at a coefficient and say "more of this feature gives a higher price" like with linear regression on the original features. The loadings (how much each original feature contributes to a component) give some idea of what a component represents, but not as directly.

### 8. Förklara vad nedanstående kod gör.

In [1]:
import numpy as np
from sklearn.decomposition import PCA

np.random.seed(42)
X = np.random.rand(1000, 3)
print(X[0:5])

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])

X3D_inv = pca.inverse_transform(X2D)

print(np.allclose(X3D_inv, X))

[[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]
 [0.05808361 0.86617615 0.60111501]
 [0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497]]
[[ 0.45459779  0.05941797]
 [-0.36808474 -0.19497059]
 [ 0.28700506 -0.14996497]
 [-0.33376092  0.61290315]
 [-0.26264222 -0.10302082]]
False


What it does step by step:
1. `np.random.rand(1000, 3)` creates a dataset with 1000 rows and 3 random features between 0 and 1. I added `np.random.seed(42)` so the numbers are the same every time it's run.
2. `PCA(n_components=2)` sets up PCA to reduce to 2 dimensions.
3. `pca.fit_transform(X)` finds the principal components and projects the data onto the first 2, so `X2D` has shape (1000, 2).
4. `pca.inverse_transform(X2D)` maps the 2D data back to 3D.
5. `np.allclose(X3D_inv, X)` checks if the reconstructed data equals the original, and prints `False`.

The point: going 3D → 2D → 3D isn't lossless. The variance along the third principal component was thrown away in the reduction, so `inverse_transform` only gives an approximation of the original data. Random uniform data has spread in all three directions, so quite a lot is lost here.

### 9. Genomför en PCA på “car_price_dataset.csv” från kapitel 3 innan du modellerar det med ML. Hur påverkas resultatet?

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("data/car_price_dataset.csv", sep=";")
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


The file is semicolon-separated, so it needs `sep=";"`, otherwise pandas reads each row as a single column.

In [3]:
df = pd.get_dummies(df, columns=["Brand", "Model", "Fuel_Type", "Transmission"], drop_first=True)

X = df.drop(columns="Price")
y = df["Price"]
print(X.shape)

(10000, 48)


I one-hot encoded the nominal columns (Brand, Model, Fuel_Type, Transmission), which increases the number of features from 9 to 48, mostly because Model has 30 unique values.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Scaling is important before PCA. PCA picks directions based on variance, so without scaling a feature like Mileage (values in the hundreds of thousands) would dominate a feature like Doors (values 2-5) just because of its larger numbers.

In [5]:
for name, model in [("Linear Regression", LinearRegression()), ("Random Forest", RandomForestRegressor(random_state=42))]:
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"NO PCA - {name}: RMSE={rmse:.2f} R2={r2:.4f}")

NO PCA - Linear Regression: RMSE=64.91 R2=0.9995
NO PCA - Random Forest: RMSE=544.92 R2=0.9677


Without PCA, linear regression gets R² 0.9995 (RMSE about 65) and random forest R² 0.968 (RMSE about 545) on the test set. Price seems to be close to a linear function of the features in this dataset, which looks synthetic.

In [6]:
for n_comp in [0.90, 0.95, 0.99]:
    pca = PCA(n_components=n_comp)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    print(f"n_components={n_comp} -> {X_train_pca.shape[1]} components, {pca.explained_variance_ratio_.sum():.3f} variance explained")

    for name, model in [("Linear Regression", LinearRegression()), ("Random Forest", RandomForestRegressor(random_state=42))]:
        model.fit(X_train_pca, y_train)
        pred = model.predict(X_test_pca)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)
        print(f"  PCA({n_comp}) - {name}: RMSE={rmse:.2f} R2={r2:.4f}")

n_components=0.9 -> 33 components, 0.919 variance explained
  PCA(0.9) - Linear Regression: RMSE=1607.02 R2=0.7189
  PCA(0.9) - Random Forest: RMSE=1148.89 R2=0.8563
n_components=0.95 -> 35 components, 0.959 variance explained
  PCA(0.95) - Linear Regression: RMSE=870.75 R2=0.9175
  PCA(0.95) - Random Forest: RMSE=937.60 R2=0.9043
n_components=0.99 -> 38 components, 0.995 variance explained
  PCA(0.99) - Linear Regression: RMSE=65.18 R2=0.9995
  PCA(0.99) - Random Forest: RMSE=866.49 R2=0.9183


PCA didn't improve the scores on this split. Keeping 90% of the variance (33 components) or 95% (35 components) made both models clearly worse. At 99% (38 components) linear regression was back at about the same result as without PCA, but random forest still lost accuracy (R² 0.918 vs 0.968).

A likely reason: PCA chooses directions with high variance in the features without looking at Price, so it can drop directions that are useful for predicting Price. For random forest, the rotated components may also make it harder to find good splits than the original columns, which are mostly 0/1 dummy variables.

These are comparisons on a single held-out split. I'm only comparing the variants here, not choosing a final model, but if I used these results to pick a configuration they would be validation results, and I would need a separate test set for a final performance estimate.